In [0]:
# Leer la tabla que acabas de subir al catálogo
df = spark.read.table("muestra_tweets")

# Ver cuántas filas tiene tu muestra
print(f"Total de filas en la muestra: {df.count()}")

# Ver las primeras filas para confirmar que todo está bien
display(df.limit(5))

Total de filas en la muestra: 1000000


tweetid,userid,tweet_time,is_retweet,quote_count,like_count,retweet_count,hashtags,urls,user_mentions,BoW,info_op
8bf72a848c6d59d0624d207114d49fcff0e4d03134323e07d2840a4242466d4018a1999082f53508b8d5a6bc0ae8fc30d5b15ab33a2111b6e7f6eb848bb8cbb7,cd65009d7512ed0a2d71a66ae438f87f7afec86b2c9a663364af8de0de32675039f65b8fd93a99da15dc62afd7e851ffb1cdb18ffb31f0d9ee8951e37f67b446,2020-07-24T09:05:40.000Z,0,0.0,0.0,0.0,0,1,0,"['exclusive', 'uk', 'home', 'test', 'dog', 'delay', 'undermine', 'push', 'reduce', 'virus', 'transmission']",0
02300bf1d1fe92f2fd711938686b922b2d364ad4c71aab173c65fd10c38d5adf2ff2ef5d411807091e07dd90d35e9856fcaea240f123381f16792de58974e05a,cd65009d7512ed0a2d71a66ae438f87f7afec86b2c9a663364af8de0de32675039f65b8fd93a99da15dc62afd7e851ffb1cdb18ffb31f0d9ee8951e37f67b446,2020-07-24T06:02:38.000Z,0,0.0,0.0,0.0,0,1,1,"['watch', 'face', 'cover', 'stop', 'spread', 'coronavirus', 'itv', 'news']",0
aace9b5b4afac7757112445d532dc4afe7fb6c512f9b3ae2b949838bf4acd4722703461c6f6c5a97be6c8341684cfc1d59d9dd6f5392655fb7d06d3e916d06d9,cd65009d7512ed0a2d71a66ae438f87f7afec86b2c9a663364af8de0de32675039f65b8fd93a99da15dc62afd7e851ffb1cdb18ffb31f0d9ee8951e37f67b446,2020-07-24T04:39:21.000Z,0,0.0,0.0,0.0,0,1,1,"['year', 'cycle', 'floods', 'sacred', 'shrines', 'locusts']",0
ca624ff447708d3ca47f67fe8158cb327408c0d5c0bbe96dfa0482207816d4c76f8303be9e6b119df09817f38bb48209391d4f26fdbe310df670a07a029a8ed9,cd65009d7512ed0a2d71a66ae438f87f7afec86b2c9a663364af8de0de32675039f65b8fd93a99da15dc62afd7e851ffb1cdb18ffb31f0d9ee8951e37f67b446,2020-07-24T04:34:03.000Z,0,4.0,0.0,4.0,0,1,1,"['online', 'tool', 'help', 'speed', 'autism', 'diagnosis']",0
7fe7af03309659324c747713a7b697fda54aed039b4528f78fe1239ac50d182b4cc5d940dfa55f8d5f5d14353c3a636419c8ac889cfdfcb25f8a51ab4c8d2770,cd65009d7512ed0a2d71a66ae438f87f7afec86b2c9a663364af8de0de32675039f65b8fd93a99da15dc62afd7e851ffb1cdb18ffb31f0d9ee8951e37f67b446,2020-07-24T04:30:22.000Z,0,0.0,0.0,0.0,0,1,0,"['bbc', 'news', 'coronavirus', 'new', 'face', 'cover', 'rule', 'come', 'force', 'england']",0


In [0]:
from pyspark.sql.functions import col, when, size, regexp_replace, split

# 1. Recargamos la tabla original DESDE CERO (esto ya crea un DataFrame nuevo y limpio)
df = spark.read.table("muestra_tweets")

# 2. Limpiamos las listas usando texto (sin from_json)
# Convertimos a string, quitamos corchetes y comillas, y separamos por coma
df = df.withColumn(
    "user_mentions", 
    split(regexp_replace(col("user_mentions").cast("string"), r"[\[\]']", ""), r",\s*")
)
df = df.withColumn(
    "hashtags", 
    split(regexp_replace(col("hashtags").cast("string"), r"[\[\]']", ""), r",\s*")
)

# 3. Filtrar filas útiles
df_grafo = df.filter(
    (size(col("user_mentions")) > 0) | (col("is_retweet") == 1)
)

# 4. Limpiar nulos y tipos
df_grafo = df_grafo.fillna(0, subset=["retweet_count", "quote_count", "like_count"])
df_grafo = df_grafo.withColumn("is_retweet", 
    when(col("is_retweet").cast("int").isNull(), 0)
    .otherwise(col("is_retweet").cast("int"))
)

# 5. Eliminar duplicados
df_grafo = df_grafo.dropDuplicates(["tweetid"])

# 6. Verificación
print(f"Filas originales: {df.count()}")
print(f"Filas útiles para el grafo: {df_grafo.count()}")

# 7. Mostrar muestra
display(df_grafo.select("userid", "user_mentions", "hashtags").limit(5))

Filas originales: 1000000
Filas útiles para el grafo: 1000000


userid,user_mentions,hashtags
80b431ee2e3cb08bdbdbc98ed43d486c34b7d17338895ec81d78308c0eb76403b8869ae47d4e64a5c1a6e1806077ecd6ace703fb61450851cb3e546e311d5148,List(2),List(0)
80b431ee2e3cb08bdbdbc98ed43d486c34b7d17338895ec81d78308c0eb76403b8869ae47d4e64a5c1a6e1806077ecd6ace703fb61450851cb3e546e311d5148,List(1),List(0)
80b431ee2e3cb08bdbdbc98ed43d486c34b7d17338895ec81d78308c0eb76403b8869ae47d4e64a5c1a6e1806077ecd6ace703fb61450851cb3e546e311d5148,List(1),List(0)
80b431ee2e3cb08bdbdbc98ed43d486c34b7d17338895ec81d78308c0eb76403b8869ae47d4e64a5c1a6e1806077ecd6ace703fb61450851cb3e546e311d5148,List(1),List(0)
80b431ee2e3cb08bdbdbc98ed43d486c34b7d17338895ec81d78308c0eb76403b8869ae47d4e64a5c1a6e1806077ecd6ace703fb61450851cb3e546e311d5148,List(1),List(0)


In [0]:
# 1. Cargar las tablas
df_tweets = spark.read.table("muestra_tweets")
df_users = spark.read.table("all_users_hashed")

# 2. Renombrar columnas conflictivas en la tabla de tuits ANTES del JOIN
df_tweets = df_tweets.withColumnRenamed("BoW", "tweet_BoW")
df_tweets = df_tweets.withColumnRenamed("info_op", "tweet_info_op") # <-- Renombramos esta

# 3. Hacer el JOIN
df_join = df_tweets.join(df_users, on="userid", how="inner")

# 4. GUARDAR LA TABLA MAESTRA (Ahora ya no debería chocar)
df_join.write.mode("overwrite").saveAsTable("datos_maestros_reto4")
print("¡Tabla 'datos_maestros_reto4' guardada con éxito!")

# 5. Verificar que se guardó bien
display(spark.read.table("datos_maestros_reto4").limit(3))

¡Tabla 'datos_maestros_reto4' guardada con éxito!


userid tweetid tweet_time is_retweet quote_count like_count retweet_count hashtags urls user_mentions tweet_BoW tweet_info_op user_reported_location user_profile_description follower_count following_count BoW retweet_ratio english_tweet_proportion earliest_tweet_time latest_tweet_time average_tweet_time median_tweet_time tweet_count stddev_tweet_time mode_0 mode_1 mode_2 mode_3 mode_4 mode_5 mode_6 mode_7 mode_8 mode_9 mode_10 mode_11 mode_12 mode_13 mode_14 mode_15 mode_16 mode_17 mode_18 mode_19 mode_20 mode_21 mode_22 mode_23 avg_tweets_per_week avg_tweets_per_day avg_tweets_per_hour avg_tweets_per_min avg_quote_count avg_like_count avg_retweet_count avg_hashtags avg_urls avg_user_mentions info_op 95c6c8282ddbf677c898990f420c6b5f94e188150be3fea0a407373c10317080712a34905dc7f41ea5ee516914356ae559a0c42be3f3e3c4cb83dcc55ea4a3f3 9e2bd3a992335e4f95e4b751c1169a023e292b543ec9b4b4f27c784dc96b4bdff7472a07d9a2e68355daad7aa778e567005d935880bbc201551bfbaf57627580 2020-12-03T04:56:52.000Z 0 0.0 0.0 0.0 0 0 1 ['lemme', 'simp'] 0 0 1 577.0 2167.0 ['fuk', 'food', 'home', 'future', 'commissioners', 'treat', 'musician', 'like', 'future', 'hard', 'live', 'music', 'cred', 'scaredy', 'wolf', 'play', 'solo', 'professional', 'prison', 'bought', 'bookshelf', 'hold', 'stuff', 'exhoing', 'streamlabs', 'wtf', 'oh', 'man', 'suck', 'hope', 'sushi', 'car', 'properly', 'work', 'soon', 'ikr', 'crying', 'saw', 'car', 'key', 'solo', 'phasmophobia', 'today', 'pm', 'est', 'ph', 'hour', 'hopefully', 'echo', 'time', 'play', 'mmmmmmmmmmmmmm', 'year', 'record', 'line', 'persona', 'new', 'outfit', 'change', 'hairstyle', 'bit', 'gt', 'lt', 'cute', 'persona', 'love', 'outfit', 'ruh', 'ro', 'feel', 'discomfort', 'hand', 'like', 'pain', 'uncomfortable', 'new', 'sticker', 'finally', 'available', 'preorder', 'ot', 'preorders', 'close', 'stic', 'omg', 'love', 'ur', 'hoodie', 'oh', 'yeah', 'rig', 'model', 'new', 'winter', 'hoodie', 'draw', 'amazing', 'hello', 'slot', 'vtuber', 'logo', 'commissions', 'interested', 'january', 'later', 'deadline', 'usd', 'ill', 'waiting', 'late', 'day', 'firsts', 'yumeweek', 'build', 'rocket', 'lot', 'fun', 'stream', 'today', 'vtubers', 'fanart', 'stream', 'thank', 'join', 'vtu', 'hit', 'follower', 'twitch', 'think', 'fix', 'stream', 'issue', 'solo', 'prison', 'professional', 'phasmophobia', 'run', 'huge', 'coward', 'think', 'fix', 'mic', 'stream', 'set', 'possibly', 'record', 'stream', 'need', 'desk', 'aha', 'looking', 'genshin', 'reddit', 'thanks', 'follow', 'help', 'great', 'ton', 'extremely', 'grateful', 'raffle', 'thanks', 'hoenn', 'stargaze', 'week', 'learn', 'oomf', 'lol', 'think', 'learn', 'year', 'change', 'failure', 'pt', 'congrats', 'starter', 'choice', 'johto', 'lol', 'minecraft', 'skin', 'actually', 'funnier', 'fanart', 'light', 'elliot', 'page', 'come', 'gonna', 'remind', 'guy', 'basic', 'basic', 'trans', 'etiquette', 'thread', 'elliot', 'page', 'isn', 'fucking', 'fictional', 'character', 'shit', 'like', 've', 'lose', 'lesbian', 'start', 'deadnaming', 'th', 'end', 'model', 'lags', 'stream', 'lot', 'hopefully', 'lag', 'way', 'ponkchamps', 'cute', 'send', 'good', 'vibe', 'finally', 'chara', 'redesign', 'time', 'lmao', 'good', 'luck', 'bad', 'luck', 'appointment', 'fingerprint', 'scan', 'job', 'check', 'bg', 'stuff', 'urgegrbbr', 'stream', 'schedule', 'logo', 'start', 'screen', 'happy', 'birthday', 'lovely', 'day', 'today', 'actually', 'gen', 'bully', 'fukin', 'hard', 'emily', 'ruining', 'career', 'hope', 'use', 'yuku', 'yo', 'paid', 'model', 'im', 'crying', 'bug', 'persona', 'audition', 'want', 'current', 'model', 'new', 'right', 'giveaway', 'hellu', 'away', 'loooong', 'duck', 'sweater', 'lucky', 'random', 'winner', 'rules', 'rt', 'comment', 'emo', 've', 'notify', 'ghostie', 'pin', 'knock', 'nauseous', 'know', 'people', 'purchas', 'rt', 'pull', 'zhongli', 'proven', 'scientist', 'retweet', 'good', 'luck', 'look', 'beautiful', 'gonna', 'mmm', 'flower', 'blue', 'time', 'lmao', 'remind', 'single', 'waking', 'day', 'bad', 'accident', '

In [0]:
# 1. Crear un Volumen en Unity Catalog (usando el catálogo 'workspace' por defecto)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.reto4_export")

# 2. Definir la ruta del Volumen (¡Esta ruta sí está permitida!)
ruta_volumen = "/Volumes/workspace/default/reto4_export"

# 3. Cargar tu tabla maestra
df_final = spark.read.table("datos_maestros_reto4")

# 4. Exportar a Parquet (para S3)
# Usamos coalesce(1) para que sea un solo archivo y no una carpeta con muchos pedazos
df_final.coalesce(1).write.mode("overwrite").parquet(f"{ruta_volumen}/datos_reto4.parquet")

# 5. Exportar a CSV (para Neo4j)
df_grafo_export = df_final.select("userid", "follower_count", "retweet_ratio", "tweet_BoW")
df_grafo_export.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{ruta_volumen}/export_neo4j.csv")

print(f"¡Archivos guardados con éxito en: {ruta_volumen}!")

¡Archivos guardados con éxito en: /Volumes/workspace/default/reto4_export!
